# 01 — Build an inspectable local RAG baseline

**Level:** Beginner · **Estimated time:** 75–90 minutes · **Scenario:** Harborline Support

By the end, you will build and evaluate a dependency-free retrieval baseline, explain every score it produces, build a bounded context window, and make abstention a measurable product behavior.

> The goal is not to make a clever chatbot. It is to establish a baseline you can falsify before adding embeddings or an LLM.


## How to use this notebook

Work in this order: read the concept, run the deterministic code, change **one** variable, inspect the trace, and write down what changed. The model API is deliberately absent: the learning objective is to understand the evidence system that an LLM would depend on.

**Scenario.** You are building a small, internal assistant for Harborline, a fictional SaaS company. Support needs trustworthy answers about customer communication and production escalation. The corpus is intentionally tiny so every result can be inspected.


## 1. What RAG changes — and what it does not

Retrieval-augmented generation (RAG) supplies external evidence at answer time. It does **not** make an answer automatically correct: ingestion can omit a document, chunking can split an answer, retrieval can rank the wrong evidence, and generation can overstate what evidence supports.

```text
OFFLINE:  documents → parse → chunks + metadata → index

ONLINE:   question → retrieve candidates → threshold policy
                                            ├─ enough evidence → bounded context → answer + citations
                                            └─ weak / empty    → abstain + next safe step
```

Keep two questions separate throughout this course:

1. **Retrieval quality:** did the system find the evidence needed to answer?
2. **Answer faithfulness:** did the final answer stay within that evidence?

This notebook evaluates the first question and implements a simple evidence boundary. The citation notebook later adds stronger provenance checks.


## 2. Why start with lexical retrieval?

Lexical retrieval ranks literal term overlap. It is weak on synonyms, but it makes failure visible: a learner can see that `payment incident` matched one source while `refund outage` did not. Dense retrieval can improve paraphrase matching later; it does not remove the need for evaluation, metadata, or an abstention policy.

The baseline has four contracts:

- **Stable identity:** every chunk has an ID and source filename.
- **Inspectable ranking:** every hit includes score, rank, and matched terms.
- **Bounded context:** only a limited amount of labelled evidence reaches the answer step.
- **Safe terminal state:** insufficient evidence returns an abstention rather than invented support.


In [ ]:
from pathlib import Path
from examples.beginner.first_local_rag import (
    EvaluationCase, build_context, evaluate_baseline, load_chunks,
    retrieve_with_trace,
)

ROOT = Path.cwd() if (Path.cwd() / 'examples').exists() else Path('../..')
DOCS = ROOT / 'examples/data/beginner-docs'
chunks = load_chunks(DOCS)
print(f'Indexed {len(chunks)} inspectable chunks from {len({c.source for c in chunks})} documents.')
[(chunk.chunk_id, chunk.source, chunk.section) for chunk in chunks[:8]]


## 3. Inspect ingestion before you trust retrieval

Ingestion is a correctness boundary. If a required policy is absent or a source has no stable ID, an apparently good answer cannot be audited. Our loader intentionally uses Markdown paragraphs as chunks. That is not a production recommendation; it is a simple unit that lets us inspect the pipeline end to end.

**Check before continuing:** locate the `Harborline Support Handbook` and its `Escalation boundary` paragraph. What information would you add before using this data in production (for example document version, tenant, access label, or updated timestamp)?


In [ ]:
question = 'Who may restart production services?'
hits = retrieve_with_trace(question, chunks, top_k=3)

for hit in hits:
    print(f'#{hit.rank} score={hit.score:.2f} matches={hit.matched_terms}')
    print(f'   {hit.chunk.chunk_id} · {hit.chunk.source} · {hit.chunk.section}')
    print(f'   {hit.chunk.text}\n')


## 4. Read a retrieval trace

The score is the fraction of query terms found in a chunk. It is **not** a calibrated probability of truth. A score of `0.60` means three of five lexical terms overlapped after stop words were removed. It says nothing about whether the passage authorizes an action or whether the text is current.

Useful debugging questions:

- Which important query term failed to match?
- Is the first hit actually responsive, or merely keyword-adjacent?
- Did a heading become a low-value retrieval candidate?
- Would an alternate phrasing retrieve a different source?


In [ ]:
paraphrase = 'Can the support team reboot checkout in production?'
for prompt in (question, paraphrase):
    trace = retrieve_with_trace(prompt, chunks, top_k=3)
    print(f'\n{prompt}')
    print([(hit.chunk.chunk_id, round(hit.score, 2), hit.matched_terms) for hit in trace])


## 5. Build context, not a document dump

Retrieval is normally followed by prompt construction. Context should be labelled, bounded, and ordered; otherwise a large or irrelevant document can hide the useful evidence, inflate latency, and make citations ambiguous.

```text
ranked hits → label with stable source IDs → apply a context budget → answer from the retained evidence
```

In a real system, the context builder would additionally apply authorization before retrieval, preserve versions and locations, and avoid placing untrusted instructions in a trusted prompt channel.


In [ ]:
context = build_context(hits, max_characters=420)
print(context)
print('\nCharacters retained:', len(context))


## 6. The abstention policy is a product decision

A baseline needs a documented answer/no-answer boundary. A weak score should not be converted into polished prose. At the same time, a threshold that is too high can hide a valid answer. Treat the threshold as a policy parameter and test it on both supported and unsupported questions.

**Failure case:** “What is the capital of France?” is outside the Harborline corpus. It should abstain even though a general-purpose model might know an answer.


In [ ]:
from examples.beginner.first_local_rag import answer

for prompt in (question, 'What is the capital of France?'):
    print(f'Q: {prompt}')
    print(answer(prompt, chunks, min_score=0.20), '\n')


## 7. Evaluate retrieval and abstention separately

A **golden set** names representative questions, the evidence IDs expected for answerable questions, and which questions should abstain. It turns a demo into an engineering baseline. Notice that retrieval success and abstention correctness can disagree: a good retriever can still use a badly tuned threshold.


In [ ]:
golden_set = [
    EvaluationCase('Who may restart production services?', ('harborline-support-7',)),
    EvaluationCase('How often do enterprise customers receive an update?', ('harborline-support-5',)),
    EvaluationCase('What must an answer distinguish?', ('harborline-policy-3',)),
    EvaluationCase('What is the capital of France?', (), should_abstain=True),
]

report = evaluate_baseline(golden_set, chunks, top_k=3, min_score=0.20)
for row in report:
    print(row)

retrieval_recall = sum(row['retrieval_hit'] for row in report if not row['abstention_correct'] or row['retrieval_hit']) / len(report)
abstention_accuracy = sum(row['abstention_correct'] for row in report) / len(report)
print(f'\nIllustrative retrieval-hit rate: {retrieval_recall:.0%}')
print(f'Abstention accuracy: {abstention_accuracy:.0%}')


## 8. Experiment: change one variable

Run the same golden set at `top_k=1` and `top_k=3`, then at thresholds `0.20`, `0.50`, and `0.80`. Do not change both at once.

Record:

| Change | Retrieval hit? | Abstention correct? | What became worse? |
| --- | --- | --- | --- |
| `top_k=1` | | | |
| `min_score=0.50` | | | |
| your chosen policy | | | |

**Interpretation:** choose a policy from representative failures, not from one impressive answer.


In [ ]:
# Your experiment: alter only one parameter in each run.
for threshold in (0.20, 0.50, 0.80):
    rows = evaluate_baseline(golden_set, chunks, top_k=3, min_score=threshold)
    correct = sum(row['abstention_correct'] for row in rows)
    print(f'threshold={threshold:.2f} · abstention accuracy={correct}/{len(rows)}')


## 9. Deliberate failure: lexical mismatch

Ask a question using words absent from the source, such as “Can support **reboot** checkout?” while the policy says “restart production services.” If the baseline misses it, that is expected evidence for a later change—not a reason to hide the failure.

Possible next interventions, in order of increasing complexity:

1. improve source wording or add synonyms where editorially correct;
2. add query expansion and evaluate whether it drifts from intent;
3. add dense or hybrid retrieval, retaining the same golden set;
4. rerank a bounded candidate set.

Do not adopt a technique just because it returns an answer; verify that it retrieves the right evidence and preserves citations.


In [ ]:
failure_prompt = 'Can support reboot checkout?'
[(hit.chunk.chunk_id, hit.score, hit.matched_terms) for hit in retrieve_with_trace(failure_prompt, chunks)]


## 10. Retrieval is not authorization

A relevant result is not necessarily a result the caller may see. Authorization must narrow the candidate corpus **before** ranking and context construction. Filtering only after a model has seen the context can leak protected data through logs, prompts, or an answer.

This small corpus has no real identity system, so we use an explicit source allow-list. Production systems generally use tenant IDs, document ACLs, source version, retention status, and caller claims in the retrieval filter. A model must not be trusted to choose its own access boundary.


In [ ]:
from examples.beginner.first_local_rag import build_context_pack, retrieve_authorized

support_only = {'harborline-support.md'}
secured_hits = retrieve_authorized(
    'Who may restart production services?', chunks, allowed_sources=support_only, top_k=3
)
secured_pack = build_context_pack(secured_hits, max_characters=320)
print('Visible IDs:', secured_pack.retained_ids)
print('Citations:', secured_pack.citations)
print('Was context truncated?', secured_pack.truncated)
print('\n' + secured_pack.text)


## 11. Measure ranking quality, not just whether a demo answers

A golden set is a small set of requests with an expected outcome and evidence. It gives you a stable contract when you change retrieval, chunking, or the corpus. This notebook uses four complementary metrics:

- **Recall@k:** at least one expected chunk is in the first `k` results.
- **Precision@k:** what fraction of returned chunks are expected evidence.
- **MRR:** how early the first expected chunk appears; an answer component benefits when evidence is near the top.
- **Abstention accuracy:** whether the system correctly withholds an answer for an unsupported request.

None measures factual correctness of generated prose. They answer a narrower question: did retrieval deliver usable evidence? The citations lesson and evaluation path test the next boundary.


In [ ]:
from examples.beginner.first_local_rag import summarize_retrieval_metrics

metrics = summarize_retrieval_metrics(report)
print(f'Recall@3:             {metrics.recall_at_k:.0%}')
print(f'Precision@3:          {metrics.precision_at_k:.0%}')
print(f'Mean reciprocal rank: {metrics.mean_reciprocal_rank:.2f}')
print(f'Abstention accuracy:  {metrics.abstention_accuracy:.0%}')

# Design question: which cost matters most for Harborline?
# Missing the current on-call rule, exposing a restricted document, and
# declining a harmless policy question have different operational costs.


## 12. Compare the baseline with BM25

Literal-overlap scoring gives every query term the same weight. BM25 adds term frequency, document length normalization, and inverse document frequency: uncommon terms are generally more discriminative than ubiquitous ones. It is still lexical; it will not magically resolve a synonym such as `reboot` versus `restart`.

The compact implementation below recomputes corpus statistics so learners can inspect it. A production search engine pre-computes them. Compare the ranking, then decide from the golden set whether the change is worth its operational complexity.


In [ ]:
from examples.beginner.first_local_rag import retrieve_bm25

for prompt in (
    'Who may restart production services?',
    'How often do enterprise customers receive an update?',
):
    overlap = [(hit.chunk.chunk_id, round(hit.score, 3)) for hit in retrieve_with_trace(prompt, chunks)]
    bm25 = [(hit.chunk.chunk_id, round(hit.score, 3)) for hit in retrieve_bm25(prompt, chunks)]
    print(f'\n{prompt}')
    print('overlap:', overlap)
    print('BM25:   ', bm25)


## 13. A decision ladder: improve the observed failure

Do not add embeddings, an agent, or a larger context window because they are fashionable. Move one step only when a measured failure justifies it.

```text
wrong / stale source? ───────> fix ownership, parsing, freshness
           │
           ├── literal synonym miss? ──> aliases, BM25, then hybrid retrieval
           ├── rule split across chunks? ──> structure-aware chunking
           ├── good candidate ranks low? ──> rerank a bounded candidate set
           ├── protected evidence appears? ──> authorization before retrieval
           └── evidence is good but answer drifts? ──> citation + faithfulness evaluation
```

A structured, live value such as an account balance is usually better served by an authenticated API or SQL query. Retrieval is useful when the answer must be synthesized from evolving unstructured evidence.


In [ ]:
# A compact upgrade experiment: compare an exact policy question with a paraphrase.
questions = [
    'Who may restart production services?',
    'Can support reboot checkout in production?',
    'What is the capital of France?',
]
for prompt in questions:
    trace = retrieve_with_trace(prompt, chunks, top_k=2)
    decision = 'answerable baseline' if trace and trace[0].score >= 0.20 else 'abstain / investigate failure'
    print(f'{decision:32} | {prompt}')
    print('  ', [(hit.chunk.chunk_id, round(hit.score, 2)) for hit in trace])


## 14. Build exercise: a release-worthy retrieval contract

Extend the baseline without adding a model:

1. Add a document version and `updated_at` value to each source record.
2. Add one exception rule whose meaning would be lost if it is separated from its parent rule.
3. Write four golden-set cases: a supported question, synonym failure, unsupported question, and a visibility-restricted question.
4. Choose `top_k` and threshold based on your expected costs; state the safe next action after abstention.
5. Produce a trace containing query, caller scope, source IDs, scores, retained context IDs, policy decision, and retrieval configuration.

**Do not** claim this baseline is production-ready just because it returns citations. A citation proves which text was shown, not that the text was current, authorized, or sufficient for every claim.


## 15. Checkpoint and next step

1. Why is a lexical score not a confidence score?
2. Why must authorization happen before retrieval rather than in the answer prompt?
3. Which metric identifies a false abstention versus a failed retrieval?
4. What evidence would justify choosing hybrid search for the `reboot` failure?
5. Give one use case where a typed tool is safer than RAG.

**Next:** in the chunking lab, change the units that retrieval can return and measure how boundary choices affect evidence coverage.

### References

- Lewis et al., [Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks](https://arxiv.org/abs/2005.11401)
- Manning, Raghavan, and Schütze, [Introduction to Information Retrieval](https://nlp.stanford.edu/IR-book/)
- Thakur et al., [BEIR](https://arxiv.org/abs/2104.08663)
- NIST, [Generative AI Profile](https://nvlpubs.nist.gov/nistpubs/ai/NIST.AI.600-1.pdf)
- [RAG explained in this repository](../../docs/what-is-rag.md)
